# Section 1 — Your First Agent (Local Ollama)

Original learning exercise covering a first LLM-based assistant, role/system prompts, input validation and a small document summary. This is a **new local starter** rather than the original Strands/Bedrock notebook. Run Code Cells 1–8 sequentially. All content is fictional; no AWS required.

## Cell 1 — Check local Ollama and qwen3:1.7b

In [ ]:
import json, platform, time, urllib.request
from pathlib import Path

BASE = "http://127.0.0.1:11434"
MODEL = "qwen3:1.7b"
with urllib.request.urlopen(BASE + "/api/tags", timeout=10) as response:
    models = json.load(response).get("models", [])
names = [m.get("name", "") for m in models]
print("Python:", platform.python_version(), "| Local models:", names)
assert MODEL in names, "Run: ollama pull qwen3:1.7b"
print("PASS: Local Ollama and Qwen3 available")

## Cell 2 — Reusable local chat function

In [ ]:
def local_chat(user_text, system_text="You are a concise, factual assistant."):
    assert isinstance(user_text, str) and user_text.strip(), "Message must not be empty"
    payload = {
        "model": MODEL,
        "stream": False,
        "think": False,
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text}
        ],
        "options": {"temperature": 0.2, "num_predict": 450},
    }
    for attempt in range(2):
        started = time.monotonic()
        request = urllib.request.Request(
            BASE + "/api/chat",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        with urllib.request.urlopen(request, timeout=210) as response:
            data = json.load(response)
        answer = (data.get("message") or {}).get("content", "").strip()
        if answer:
            return {"answer": answer, "latency_s": round(time.monotonic() - started, 3)}
        payload["options"]["num_predict"] = 700
    raise RuntimeError("Ollama returned empty content twice")

print("PASS: Chat function ready (thinking disabled)")

## Cell 3 — First assistant response

In [ ]:
first = local_chat("In two sentences, explain what an AI agent is.")
assert first["answer"]
print(first["answer"])
print("Latency:", first["latency_s"], "seconds")
print("PASS: First local response")

## Cell 4 — Prompt and role comparison

In [ ]:
response_a = local_chat("Explain AI agents in one short sentence.")
response_b = local_chat(
    "Explain AI agents in one short sentence.",
    system_text="You are a Python instructor teaching beginners. Avoid jargon."
)
assert response_a["answer"] and response_b["answer"]
print("Default:", response_a["answer"][:500])
print("Instructor:", response_b["answer"][:500])
print("PASS: System prompts tested")

## Cell 5 — Simple input guard

In [ ]:
def validate_question(text):
    if not isinstance(text, str):
        raise TypeError("Question must be text")
    text = text.strip()
    if not 1 <= len(text) <= 1000:
        raise ValueError("Question must contain 1-1000 characters")
    return text

assert validate_question("  hello ") == "hello"
try:
    validate_question("")
except ValueError:
    print("PASS: Empty input rejected")
else:
    raise AssertionError("Missing empty-input rejection")
print("PASS: Local input validation")

## Cell 6 — Small document analysis

In [ ]:
fictional_document = (
    "Sample service agreement. Delivery within 10 days. "
    "Payment due 30 days after receipt of invoice. "
    "The supplier may charge a late fee after 45 days."
)
analysis = local_chat(
    "Summarise only the facts in this fictional document in three bullet points:\n"
    + fictional_document
)
assert analysis["answer"].strip()
print(analysis["answer"][:1100])
print("PASS: Local document prompt completed")

## Cell 7 — Basic response assertions

In [ ]:
results = [
    ("first_reply", first["answer"]),
    ("default_prompt", response_a["answer"]),
    ("instructor_prompt", response_b["answer"]),
    ("document_analysis", analysis["answer"]),
]
assert len(results) == 4
assert all(isinstance(answer, str) and answer.strip() for _, answer in results)
print("PASS: 4 responses exist (content accuracy needs human review)")

## Cell 8 — Save actual local evidence

In [ ]:
report = {
    "section": 1,
    "title": "Your First Agent - local introductory chat lab",
    "model": MODEL,
    "scope": "Ollama API; not Strands/Bedrock deployment",
    "responses": [{"test": name, "answer": answer} for name, answer in results],
    "pass_count": len(results),
}
output = Path("section1_local_report.json")
output.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
assert output.exists()
print("SAVED:", output.resolve())
print("PASS: Section 1 local introduction lab")